# データ・AI活用実践（初級）第03回

-------------------

## 環境データ分析： 典型的なデータの分析方法

前回は，入手したデータを単にちらっと見てみるだけでした．今回はそれっぽいデータ分析をやってみます．といっても，とても新しい内容はそれほどなく，基本的にはこれまでの授業で行ってきたことを大気環境データを相手にやってみるだけです．

今回のデータも，ネット上からダウンロードします．

In [ ]:
# ここは覚えなくても大丈夫！
csvurl <- "https://www.cc.kyoto-su.ac.jp/~ogohara/lecture/DataAI_FirstCourse/data.csv"
download.file(csvurl, destfile = "data.csv") #data.csvという名前のファイルとしてダウンロード

もし`data.csv`の在りかがわかって，ダブルクリックできるならしてみてください．ExcelかGoogleスプレッドシートで中身を確認できるかもしれません（エラーになるかもしれません）．小郷原がスクリーンやサイドディスプレイに表示しますので，できないなら無理にやらなくても全然大丈夫です．かなりたくさんの列があり，都道府県名が書かれていることがわかります．そして，１都道府県につき3列あります．平均気温の列と品質情報と均質番号なる列があります．品質情報と均質番号はなんとなくほとんどすべて整数で，8と1ですね．このようなデータファイルを使って最初にやりたいことは以下の通りです．

- 一番左の時間軸に加えて，各都道府県の平均気温列だけ読み込み，品質情報と均質番号は捨てる
- 時間データを，文字列ではなく時間データとして読み込む
- 不要な行をスキップ（もしくはあとで削除）

これらができて初めて，なんとかかんとか分析に耐えうるデータフレームが手に入るわけです．その後，やりたいことは以下の通りです．

- 那覇，名古屋，札幌の平均気温の時間平均値を求めて比べてみる
- 那覇，名古屋，札幌の平均気温のヒストグラムを1枚の図に重ね書きしてみる

盛りだくさんですが，頑張りましょう．

-----------------

### データの読み込みと整形（データラングリング）

これはいわば前処理です．本丸であるデータ分析ではありません．にもかかわらず，結構難しくて，面倒くさくて，面白くないです．ですが，世の中のデータサイエンティストの仕事の半分以上は，なぞの文字列がそこら中に混入し，使えない行や列がたくさんあり，時間データの形式が普通でなく，日本語満載の扱いづらいデータを分析しやすい形式のデータに変換する作業に費やされるそうです．つまり，データサイエンティストである以前に優秀なプログラマでなければならないわけです（ちゃんと社内で分業されているところもあるでしょう）．したがって，「いややなー」と思うのはわかりますが，できるようになりましょう．

まずは前回と同じく，tidyverseというライブラリを使えるようにします．

In [ ]:
library(tidyverse)

改めて，先ほどダウンロードしたCSVファイルをreadします．

In [ ]:
df<-read_csv('data.csv', locale=locale(encoding='Shift_JIS'), skip=2)

なんだかたくさん出力が得られましたが，とりあえずエラーではないと思います．`df`に代入したので，`df`の中身を確認してみましょう．

In [ ]:
df

う，うーーん，スゲー嫌な感じですけど，とりあえず読み込めて表示できました．もちろんこれだけではだめです．那覇が3つありますし，時間軸もうまいこと西暦に変換してくれてはいるものの，まだ文字列`<chr>`です．ここからどうやって好みのデータセットに変えればいいでしょうか．時間軸はdateみたいにわかりやすいアルファベットの名前で，文字列でなく，都道府県名をすべてアルファベットに変えるのは大変ですが，せめて使う列だけ残したいですね．ヘッダでない最初の2行は意味のない日本語データなので消したいです．まずは，ヘッダ情報を確認しましょう．

In [ ]:
names(df)

ヘッダだけがたくさん出てきました．

In [ ]:
ncol(df)

はい，列（column）の数は142だとわかりました．ちなみに，以下のようにすると行数も含めてわかります．

In [ ]:
dim(df)

とりあえず，目下やることは上で表示された列名のうち，最初の要素を’date’に変え，2番目以降は3つに1つだけ残すことです．したがって，残したい列番号は以下のようになりますね．

In [ ]:
cnum <- c(1,seq(2,ncol(df), by=3))
cnum

`seq`という関数は等差数列を作成する関数だと思えばいいです．"sequence"です．初項が2，`ncol(df)`まで，3ごとに（2とばし）作成します．そうしてできた配列を1とうい要素の後ろに結合して新たな配列にしています．これを使えば，所望のヘッダが手に入ります．

In [ ]:
cname<-names(df)[cnum]
cname

とはいえ，これはまだ所望のヘッダリストではありません．`...`という文字列より右側が不要ですよね．不要部分を削除して都道府県名だけにしましょう．

In [ ]:
prenum <- unlist( strsplit(cname,"\\.\\.\\.") )

試しに中身を確認してみてください．`...`で文字列が分割されて，都道府県名と数値が混在した配列になっているはずです．`strsplit`って何？`unlist`は？とお思いでしょうが，1つ1つやっているとRのプログラミングの授業になるのでやりません．ググったら一瞬で出てくるので調べてみてください．ここから，1つ飛ばしで要素を取り出して，都道府県名だけほしいですね．

In [ ]:
prenum[ seq(1,length(prenum),by=2) ]

1から始まって，`prenum`の長さまで，2ごとに取り出しました．47都道府県分あるでしょうか？

In [ ]:
prename <-prenum[ seq(1,length(prenum),by=2) ]
length(prename)

はい，47都道府県＋時間データ1つですね．最後に，最初の要素名を変えましょう．

In [ ]:
prename[1] <- "date"
prename

やっとですが，まだ何も終わっていません．データの前処理とはこれほどまでに大変なのです．CSVファイルを読み直しましょう．

In [ ]:
df<-read_csv('data.csv', locale=locale(encoding='Shift_JIS'), col_select=all_of(cnum), skip=5, col_names=FALSE)

In [ ]:
colnames(df) <- prename
df47 <- type_convert(df, col_types=cols(date=col_date("%Y/%m")))

- `cnum`番目の列だけ使います．最初の5行をスキップしてください．そのうえで，最初の行を列名に使うのではなく，適当に名付けてください．
- `df`の列名をprenameに設定してください．
- `df`の`date`列の型をdate型に型変換してください．なお，「年/月」の形式で読んでください．

がソースコードの日本語訳です．中身を確認しましょう．

In [ ]:
df47

よっしゃーーーー！第1列は`date`という名前になり，型が`<date>`です．ほかのデータは`<dbl>`＝実数です．

やっとできました．このように，データ分析の過程でもっとも苦しいのは前処理だと私は思います．お気づきかもしれませんが，上で行ったとても苦しい前処理プログラミングは，今回使っているCSVデータにしか適用できません．ひとたび別の種類のデータ（企業の売り上げ，webの閲覧情報，SNSのツイート履歴，社会調査，株価や為替，etc）になれば，また1から考える必要があります．どのような関数をどのようなオプションとともに用いれば何が起こるか，は覚える価値があるかもしれませんが，処理内容自体を覚える価値は全くないのです．新しいデータに皆さんが直面したときは，どうすれば扱いやすいデータになるか，それを実現するにはどう書けばいいかも皆さん自身が考えることになるのです．

--------------

### 簡単な代表値の計算

大変なことはすでに終わりました．ここからがこの授業の本丸です．いどんな都道府県の2005年から2022年までの各月の平均気温が手に入ったわけですから，いろいろ計算してみたいですよね．まず，列名はどんなものだったか思い出しましょう．

In [ ]:
names(df47)

那覇と名古屋と札幌を抽出してみましょう．

In [ ]:
df47[,c("date","那覇","名古屋","札幌")]

`[]`の中の`,`が不自然ですが，`,`の左側に数字を書けば，その行のデータだけ抜き出します．,の右側に列名もしくは列名のリストを書くことでそれらの列だけ抜き出します．何も書かなければ，「全部」という意味になります．では，とにかく全期間の平均を求めましょう．

In [ ]:
summary(df47[,c("那覇","名古屋","札幌")])

．．．困りました．もう授業が終わってしまいそうです．`summary`関数は文字通り，一通りの代表値を計算してくれます．Minは最小値，Maxは最大値，Medianは中央値，Meanは平均値，1st Qu.は最初の四分位点＝25%分位点，です．これらの代表値については，「ビジネスデータ分析」の回（5回目から10回目）で習いますので，ここではスルーしましょう．最大値，最小値，平均値くらいがわかれば十分です．

例えば，ここ20年間ほどの間の，**月平均気温**の最大値は那覇と名古屋でさほど変わりません．一方，**月平均気温**の最小値は那覇と名古屋では結構違います．つまり，夏の暑さは2都市であまり変わらないけど，冬は圧倒的に那覇のほうが暖かい，ということです．直観と整合的です．ちなみに，**月平均気温**が30度を上回る状況をちゃんと理解できているでしょうか？夜も含めた気温の平均値が30度を超えるということですよ．やばいですね． 次は札幌と名古屋を比べましょう．最高気温はやはり札幌のほうがずっと低いです．夏涼しいということですね．冬だってちゃんと寒いですが，最低気温における那覇と名古屋のさほど，名古屋と札幌の差は大きくありません．札幌の25%分位点が0.4度なので，ざっくり言えば，1年の1/4は月平均気温が0度以下ってことです．

みなさんも，自分の出身地のデータを`summary`で要約してみたり，いろいろやってみてください．

----------------

## 自由課題

1. 授業をやるたびにこんな前処理をやっていては授業になりません．次回からは上の最終形態をすぐ使えるようにするために，df47をCSVファイル（pref47_temp.csv）として出力しておきましょう．**次回の授業で使用するので，確実にやっておいてください．**
なお，下に示されているソースコードを（`#`を消してから）実行すれば確かに保存されますが，自分のPCに保存されるわけではありません．それに，次回の授業時には（googleによって）削除されています．左の「ファイル」メニューから当該ファイル名を右クリックして，自分のPCに「ダウンロード」しておきましょう．

In [ ]:
#write.csv(x = df47, "pref47_temp.csv", fileEncoding="CP932" )

2. 以下のサンプル配列`test`を使って，平均値，中央値，標準偏差を求めましょう．標準偏差とは，データの散らばり具合を示す代表値です．いずれもRを電卓のように使って求める必要はなく，とても簡単な方法で求めることができるので，ググってみましょう．

In [ ]:
test <- rnorm(10, 4, 5)
test

3. ヒストグラムを書いてみよう。那覇だけでもいいし、3都市重ね書きしてもいいです。

> "ヒストグラム"という言葉自体も，「ビジネスデータ分析」の解で学習します．さらには，そのような図の意味も書き方も習っていないわけですから，「さあ描け」では無理ですよね．あらかじめソースコードを記しておきますので，自分の出身地や彼氏彼女の出身地，推しの出身地に変えていろいろ比べてみましょう．

In [ ]:
#何もしないと日本語が文字化けします．日本語フォントをインストールします．
system("apt-get install -y fonts-noto-cjk")

In [ ]:
#Noto Sans CJK JPというフォントを使います宣言
theme_set(theme_bw(base_family = "Noto Sans CJK JP"))

In [ ]:
#alphaは透明度．colはcolor．いろいろ変えて楽しみましょう．
g <- ggplot(df47) +
  geom_histogram( aes(x=札幌), position='identity', alpha=0.5, col='blue' ) +
  geom_histogram( aes(x=那覇), position='identity', alpha=0.5, col='red' ) +
geom_histogram( aes(x=名古屋), position='identity', alpha=0.5, col='green' )
plot(g)
# 環境によっては横軸が文字化けするかもしれません．